# Published Stokes experiment: archived high-order results

This notebook displays executed **pyMHM with DOLFINx/UFL local assembly**
experiments and digitized curves from
[Araya, Harder, Poza and Valentin (2017)](https://doi.org/10.1016/j.cma.2017.05.027),
Figures 3 and 4. P2/P2 local fields with constant face traces use six crisscross
meshes; P3/P3 with linear traces also use six meshes. Every local problem
occupies one triangle. These spaces are distinct from refined P1/P1 and
Taylor–Hood P2/P1.

The exact velocity follows from $\psi=-128x^2(x-1)^2y^2(y-1)^2$,
$u=(\psi_y,-\psi_x)$, and pressure is $p=150(x-1/2)(y-1/2)$.
Viscosity is one, drag zero, boundary velocity zero and mean pressure zero.
The archived solves used forcing differentiated symbolically in UFL.

All values and figures below are read from archived artifacts. No finite
element solve, stress-identity calculation or published-curve comparison is
recomputed by this notebook.

In [ ]:
from pathlib import Path
import sys

# Locate the checkout from the repository root or any notebook subdirectory.
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()
     and (path / "src" / "pymhm").is_dir()
     and (path / "notebooks").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Run this notebook from the PyMHM checkout.")
root = ROOT
for directory in (ROOT, ROOT / "examples"):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))


In [ ]:
import json
from pathlib import Path
from IPython.display import Image, JSON, display

p2 = json.loads((root / "examples/results/stokes_2017_ufl_crisscross.json").read_text())
p3 = json.loads((root / "examples/results/stokes_2017_ufl_p3.json").read_text())

display(JSON({
    "code": "pyMHM with DOLFINx/UFL local assembly",
    "P2_P2_recorded_levels": [row["n"] for row in p2["results"]],
    "P3_P3_recorded_levels": [row["n"] for row in p3["results"]],
}))

## Declared approximation and extraction tolerance

Each square is split into four triangles by its center. The maximum diameter is
$H=1/n$. Figure 3's plotted abscissas instead appear to be the shorter side
$H/\sqrt2$; Figure 4 and Table 1 use $H$. Both abscissas are retained in the
published CSVs. The comparison pairs the ordered levels with Table 1.

The stabilization is $\tau_T=m_T h_T^2/8$, where
$m_T=\min(1/3,C_T)$ and $C_T$ is the sharp local inverse-inequality constant.
A generalized eigenproblem gives $C_T=1/96$ for P2 and
$0.003353888994497343$ for P3. These constants are not fitted to the curves.
Assembly and squared errors use degree-16 quadrature.

Digitized published ordinates have a conservative 1% comparison allowance.
This is an extraction tolerance, not a statistical confidence interval.


In [ ]:
for data in (p2, p3):
    comparison = data["published_comparison"]
    display(JSON({
        "code": "pyMHM with DOLFINx/UFL local assembly",
        "local_degree": data["local_degree"],
        "trace_degree": data["trace_degree"],
        "digitized_article_figure": comparison["figure"],
        "declared_extraction_allowance": comparison["relative_digitization_tolerance"],
        "recorded_maximum_relative_discrepancy": comparison["maximum_relative_discrepancy"],
    }))

In [ ]:
for degree in (2, 3):
    path = root / f"docs/figures/reproduction/stokes/crisscross-p{degree}/stokes-2017-convergence.png"
    display(Image(filename=str(path), width=1100))

## Exact and numerical pressure

Each row below uses a common scale for the analytical and numerical pressure.
The third panel shows sampled pointwise error on broken local plotting meshes.
Changing P2/P2 to P3/P3 also changes trace degree, so this comparison does not
isolate the effect of the local degree alone.


In [ ]:
for degree in (2, 3):
    path = root / f"docs/figures/reproduction/stokes/crisscross-p{degree}/stokes-2017-pressure-n16.png"
    display(Image(filename=str(path)))


## Preserve the unresolved stress diagnostic

The article defines $\sigma=-\nabla u+pI$. Its full stress error satisfies

$$\|\sigma_h-\sigma\|^2=\|\nabla e_u\|^2+2\|e_p\|^2
-2(e_p,\nabla\cdot e_u).$$

The full stress L2 curve does not agree with Figure 3. At the coarsest mesh it
is 5.69316, while the digitized value is 3.53636. The latter is close to the
velocity gradient error. Direct degree-18 integration verifies the identity;
we keep the full stress definition and record the disagreement.

Equation (8) uses domain-diameter weighting in conforming H(div), whereas the
numerical section names a broken norm without restating the weight. Figure 3
agrees with weight one. Both conventions are saved and plotted; a formal
contradiction between their definitions is not asserted. The P3/P3 stress H(div) curve still differs by up to 9.87%.

The identity concerns our reconstructed fields and does not identify a bug in
the original article. Orthogonal projection of the forcing onto P2 can only
reduce the divergence residual, so it cannot explain the larger published
P3/P3 H(div) ordinate.


In [ ]:
identity = p2["stress_identity_independent_quadrature_check"]
display(JSON({
    "code": "pyMHM with DOLFINx/UFL local assembly",
    "recorded_independent_quadrature_check": identity,
}))
for data in (p2, p3):
    display(JSON({
        "local_degree": data["local_degree"],
        "recorded_stress_norms": [
            {key: row[key] for key in (
                "n", "pseudostress_l2", "pseudostress_hdiv_broken", "pseudostress_hdiv_domain_scaled"
            )}
            for row in data["results"]
        ],
    }))

## Archived data and scope

The numerical records are `examples/results/stokes_2017_ufl_crisscross.json`
and `examples/results/stokes_2017_ufl_p3.json`. The digitized published values
are in `examples/results/published/araya2017_figure3.csv` and
`examples/results/published/araya2017_figure4.csv`. The displayed PNG figures
are stored under `docs/figures/reproduction/stokes/`.

The full method, data provenance and scientific qualifications are described in
[the reproduction case page](../../../docs/cases/reproduction.md). Displaying these
records does not independently reproduce the original article's implementation,
and no timing claim is inferred from the archived accuracy runs.